### nb_02 — ブロンズ層 → シルバー層（オントロジーのバインド先）

**アタッチするレイクハウス**

| 役割 | レイクハウス | 設定 |
|---|---|---|
| 書き込み先 | `lh_its_asset_silver` | **必ず「既定」に設定する** |
| 読み取り元 | `lh_its_asset_bronze` | 追加のレイクハウスとしてアタッチ |

テーブル参照は3階層（`<レイクハウス>.<スキーマ>.<テーブル>`）です。既定のレイクハウス
（＝書き込み先）のテーブルは修飾なしで書けますが、**追加アタッチしたレイクハウスを
読むときは必ず修飾が必要**です。次のセルの `BRONZE_LAKEHOUSE` / `BRONZE_SCHEMA` を
環境に合わせて設定してください。

| | |
|---|---|
| 入力 | `lh_its_asset_bronze` の `bz_*` テーブル（全取り込み日を含む） |
| 出力 | `lh_its_asset_silver` の `sv_*` テーブル（「現在の姿」に整えたもの） |
| | `dq_silver_status`（毎回出力する鮮度・状態の一覧） |
| | `dq_<entity>_duplicates`（業務キーが重複したときのみ出力する隔離テーブル） |

#### シルバー層はオントロジーのエンティティと1対1

`ENTITIES` に並ぶ17本は、そのままオントロジーのエンティティ型17個に対応します。
`nb_03` がバインドするのはこの17本で、名前も1対1です。

そのため、このノートブックは実行のたびに **`ENTITIES` の定義とシルバー層の
テーブル一覧を一致させます**。定義から外れた `sv_*` が残っていると、オントロジーが
バインドしていないテーブルがシルバー層に並び、どれが正なのか分からなくなります。

ブロンズ層についても、必要なテーブルが揃っているかを事前に検査します。ここには
`ENRICHMENTS` の結合相手も含まれます（`sv_person` は `bz_entra_id_person` と
`bz_successfactors_employee_profile` の2本から作られます）。

#### 解いている問題

ブロンズ層は取り込み日ごとにデータが積み上がります。人事システムから毎日フルダンプが
届けば、同じ `PersonId` の行が日数分だけ存在します。

一方オントロジーのエンティティ型キーは「各レコードを一意に識別するもの」であり、
同じキーが複数行ある状態はこの前提を壊します。**しかもエラーにはならず、「1人が2人に
見える」という形で静かに結果が狂います。**

#### エンティティ型キーは業務キーから生成する（重要）

ソース由来のサロゲートキー（`PersonSkillId`、`AvailabilityId` など）をオントロジーの
キーにしてはいけません。ソースが再エクスポートすると振り直されることがあり、その場合
次の問題が起きます。

- 業務的には何も変わっていないのに、オントロジーから見ると別の実体に見える
- 別の組み合わせに同じ ID が付けば、キーが衝突する

そこで、業務キー（`PersonId` + `SkillId` など）から決定論的にキーを生成します。

```
PersonSkillKey = sha2("PersonId|SkillId")
```

こうすると業務キーで重複排除している限り、キーは構造的に重複しえません。日をまたいでも
同じ実体は常に同じキーを持ちます。ソース由来の ID は参考情報として列に残します。

#### 品質チェックの考え方

チェックは「処理を止めるため」ではなく **「壊れたものを公開しないため」** に行います。
例外は投げません。またキーを生成にしたことで、日常の取り込みで警告は出ません。

WARN が出るのは業務キー自体が重複しているとき、つまり**本当にソース側の実データが
異常なとき**だけです。だから WARN は「人が見るべき事象」を意味します。

#### 設定

`BRONZE_SCHEMA` は環境に合わせてください。スキーマ有効レイクハウスなら `'dbo'`、
スキーマ無効（レガシー）なら `None` です。

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ブロンズ層のレイクハウス名とスキーマ名。
# Fabric のスキーマ有効レイクハウスでは、テーブル参照が3階層になります。
#   <レイクハウス名>.<スキーマ名>.<テーブル名>   例: lh_its_asset_bronze.dbo.bz_entra_id_person
# 既定のスキーマは dbo です。スキーマ無効（レガシー）のレイクハウスの場合は
# BRONZE_SCHEMA = None にすると2階層（<レイクハウス名>.<テーブル名>）で参照します。
BRONZE_LAKEHOUSE = "lh_its_asset_bronze"
BRONZE_SCHEMA = "dbo"


def bronze_ref(table_name):
    """ブロンズ層テーブルの完全修飾名を返す。"""
    if BRONZE_SCHEMA:
        return f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}.{table_name}"
    return f"{BRONZE_LAKEHOUSE}.{table_name}"


# 鮮度の警告しきい値（日）。_valid_as_of がこれより古いテーブルは STALE とする
STALE_AFTER_DAYS = 3

# ENTITIES の定義から外れたシルバー層テーブル（sv_* / dq_*）を削除するか。
# False にすると、対象を表示するだけで削除しません。
DROP_ORPHAN_TABLES = True

#### 取り込みモードとエンティティの定義

エンティティごとに、ブロンズ層のどの範囲を採用するかが違います。

| モード | 対象 | 動作 |
|---|---|---|
| `snapshot` | 毎回フルダンプが届くもの | 最新の取り込み日だけを採用。前回あって今回ない行は削除されたとみなす |
| `append` | 記録が増えていくもの | 全取り込み日を対象にし、同じキーが再送された場合のみ最新を採用 |
| `timeseries` | 業務上の時間軸を持つもの | 業務キー＋時間軸ごとに最新の取り込み分を採用 |

あわせて `ENRICHMENTS` で、**ソースをまたぐ結合**を宣言します。ブロンズ層18本に対して
シルバー層が17本なのは、`sv_person` がブロンズ層2本の結合で作られるためです。
Entra ID は氏名・部署名・役職・勤務地・メールを持ちますが、組織コード・勤続年数・
雇用区分は持ちません（人事システムの領域）。ブロンズ層は届いたまま分けて置き、
シルバー層で業務の単位に組み立て直します。

In [ ]:
# ■ 3つの取り込みモード
#   snapshot   … 毎回フルダンプが届く。最新の取り込み日だけを採用する。
#                 前回あって今回ない行は「削除された」とみなし、シルバー層から消える。
#   append     … 記録が増えていく。全取り込み日を対象にし、
#                 同じキーが再送された場合のみ最新を採用する。
#   timeseries … 業務上の時間軸（YearMonth など）を持つ。全取り込み日を対象にし、
#                 業務キー＋時間軸ごとに最新の取り込み分を採用する。
#
# silver_table: (bronze_table, mode, 業務キー, エンティティ型キー名, キーを生成するか)
#   生成する    … 関連実体。ソースのIDが振り直される前提で、業務キーから生成する
#   生成しない  … 業務キーがそのまま安定した識別子（PersonId, ProjectId など）
ENTITIES = {
    # --- 人材資産（3システムに分かれている） ---
    "sv_person":                 ("bz_entra_id_person",                      "snapshot",   ["PersonId"],                    "PersonId",               False),
    "sv_organization":           ("bz_successfactors_organization",          "snapshot",   ["OrganizationId"],              "OrganizationId",         False),
    "sv_person_skill":           ("bz_successfactors_person_skill",          "snapshot",   ["PersonId", "SkillId"],         "PersonSkillKey",         True),
    "sv_person_certification":   ("bz_successfactors_person_certification",  "snapshot",   ["PersonId", "CertificationId"], "PersonCertificationKey", True),
    "sv_availability":           ("bz_d365_project_operations_availability", "timeseries", ["PersonId", "YearMonth"],       "AvailabilityKey",        True),
    "sv_assignment":             ("bz_d365_project_operations_assignment",   "snapshot",   ["PersonId", "ProjectId"],       "AssignmentKey",          True),
    # --- プロジェクト資産 ---
    "sv_project":                ("bz_servicenow_project",                   "snapshot",   ["ProjectId"],                   "ProjectId",              False),
    "sv_project_technology":     ("bz_servicenow_project_technology",        "snapshot",   ["ProjectId", "TechnologyId"],   "ProjectTechnologyKey",   True),
    "sv_project_required_skill": ("bz_servicenow_project_required_skill",    "snapshot",   ["ProjectId", "SkillId"],        "ProjectRequiredSkillKey", True),
    # --- 顧客資産 ---
    "sv_customer":               ("bz_salesforce_customer",                  "snapshot",   ["CustomerId"],                  "CustomerId",             False),
    "sv_stakeholder":            ("bz_salesforce_stakeholder",               "snapshot",   ["StakeholderId"],               "StakeholderId",          False),
    "sv_opportunity":            ("bz_salesforce_opportunity",               "snapshot",   ["OpportunityId"],               "OpportunityId",          False),
    # --- 成果物・ナレッジ（積み上がる） ---
    "sv_deliverable":            ("bz_sharepoint_online_deliverable",        "append",     ["DeliverableId"],               "DeliverableId",          False),
    "sv_knowledge":              ("bz_confluence_knowledge",                 "append",     ["KnowledgeId"],                 "KnowledgeId",            False),
    # --- マスタ ---
    "sv_skill":                  ("bz_dataverse_skill",                      "snapshot",   ["SkillId"],                     "SkillId",                False),
    "sv_certification":          ("bz_dataverse_certification",              "snapshot",   ["CertificationId"],             "CertificationId",        False),
    "sv_technology":             ("bz_dataverse_technology",                 "snapshot",   ["TechnologyId"],                "TechnologyId",           False),
}

AUDIT_COLS = ["_source_system", "_source_entity", "_source_file", "_ingested_at", "ingest_date"]

# 鮮度を業務プロパティ（DataAsOfDate）として持たせるテーブル。
# 「その情報はいつ時点か」が業務判断に影響するものだけに限定する。
# 稼働情報は「2026年9月に30%空いている」の答えが何日時点かで意味が変わるため対象。
# 一方、人の氏名や顧客の業界は鮮度を気にされないので対象外。
EXPOSE_AS_OF_DATE = {"sv_availability"}

# ■ ソースをまたぐ結合（ENRICHMENTS）
# ブロンズ層は「どのシステムから何が届いたか」をそのまま写す層なので、
# 1つの業務実体が複数システムに分かれていても、分かれたまま置く。
# それを業務の単位に組み立て直すのがシルバー層の仕事。
#
# Person がその例。Entra ID は表示名・部署名・役職・勤務地・メールを持つが、
# 組織コード・勤続年数・雇用区分は持たない（人事システムの領域）。
# そこで人事側の従業員マスタを左結合して、1つの Person に戻す。
#
# silver_table: [(結合するブロンズ表, 結合キー), ...]
ENRICHMENTS = {
    "sv_person": [("bz_successfactors_employee_profile", ["PersonId"])],
}

# 結合後の列順。オントロジーのバインドを安定させるため、明示的に固定する。
COLUMN_ORDER = {
    "sv_person": ["PersonId", "FullName", "OrganizationId", "OrganizationName",
                  "JobTitle", "YearsOfService", "WorkLocation", "EmploymentType", "Email"],
}

#### 事前確認

ブロンズ層のテーブルが参照できるかを先に確かめます。ここで落ちる場合、原因はほぼ
「スキーマ指定」か「レイクハウス未アタッチ」です。

In [ ]:
# --- 事前確認: ブロンズ層のテーブルが参照できるか ---
# ここで落ちる場合、原因はほぼ「スキーマ指定」か「レイクハウス未アタッチ」です。
# 必要なブロンズ層テーブル = エンティティの主テーブル + 結合相手（ENRICHMENTS）
primary = {t: s for s, (t, *_rest) in ENTITIES.items()}
enrich = {b: s for s, joins in ENRICHMENTS.items() for (b, _k) in joins}
required = list(primary) + [b for b in enrich if b not in primary]

missing = [t for t in required if not spark.catalog.tableExists(bronze_ref(t))]

print(f"必要なブロンズ層テーブル: {len(required)} 本"
      f"（主テーブル {len(primary)} + 結合相手 {len(required) - len(primary)}）")
print(f"シルバー層で作るテーブル: {len(ENTITIES)} 本"
      f" ← オントロジーのエンティティ型と1対1")

if missing:
    print(f"\n[ERROR] ブロンズ層のテーブルが {len(missing)} 件見つかりません。")
    print(f"        参照名の形式: {bronze_ref(missing[0])}")
    print()
    print("  見つからないテーブルと、それが必要な理由:")
    for t in missing:
        if t in primary:
            print(f"    - {t}  →  {primary[t]} の主テーブル")
        else:
            print(f"    - {t}  →  {enrich[t]} に結合する相手")
    print()
    print("  確認してください:")
    print("  1. nb_01_ingest_bronze を実行済みか")
    print("  2. このノートブックに lh_its_asset_bronze をアタッチしているか")
    print("     （エクスプローラー左の「レイクハウス」→「+」から追加）")
    print("  3. スキーマ有効レイクハウスなら BRONZE_SCHEMA = 'dbo'、")
    print("     スキーマ無効（レガシー）なら BRONZE_SCHEMA = None に設定する")
    print()
    print("  現在アタッチされているテーブル:")
    try:
        for t in spark.catalog.listTables():
            print(f"    - {t.name}")
    except Exception as e:
        print(f"    (一覧を取得できませんでした: {e})")
    raise SystemExit("ブロンズ層のテーブルを参照できないため中断します")

#### シルバー層を作る

各エンティティについて、対象日の絞り込み → 重複の検証 → 最新行の選別 →
エンティティ型キーの生成 → 鮮度列の付与 → 公開可否の判定、の順に処理します。

**検証は書き込みの前に行います。** 業務キーが重複しているテーブルは公開せず、
重複行を `dq_<entity>_duplicates` に隔離したうえで、前回のシルバー層を残します。

In [ ]:
def dedup_latest(df, business_key):
    """業務キーごとに、最も新しい取り込み分だけを残す。
    取り込み日が新しいものを優先し、同一日なら後から取り込んだ行を優先する。"""
    w = Window.partitionBy(*[F.col(c) for c in business_key]).orderBy(
        F.col("ingest_date").desc(), F.col("_ingested_at").desc()
    )
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")


report = []

for silver, (bronze, mode, business_key, entity_key, generate_key) in ENTITIES.items():
    df = spark.table(bronze_ref(bronze))
    bronze_rows = df.count()

    # --- 1. モードに応じて対象の取り込み日を絞る ---
    if mode == "snapshot":
        latest = df.agg(F.max("ingest_date").alias("d")).collect()[0]["d"]
        df = df.filter(F.col("ingest_date") == F.lit(latest))
        scope = f"latest ({latest})"
    else:
        scope = "all dates"

    # --- 2. 検証: 同一取り込み日の中で業務キーが重複していないか ---
    # 取り込み日をまたいだ重複は正常（それを次の手順で排除するのが仕事）。
    # 異常なのは「同じ日のダンプの中に同じ業務キーが2行ある」ケースで、
    # これはソース側の実データ異常（二重登録など）を意味する。
    # 重複排除の *前* に見ないと検出できないため、ここで検査する。
    same_day_dup = (
        df.groupBy(*business_key, "ingest_date")
          .count()
          .filter(F.col("count") > 1)
    )
    n_dup = same_day_dup.count()
    if n_dup > 0:
        # 隔離用に、重複していた「元の行」を排除前に取っておく
        quarantine_df = df.join(
            same_day_dup.select(*business_key, "ingest_date"),
            on=business_key + ["ingest_date"],
            how="inner",
        )

    # --- 3. 業務キーごとに最新の取り込み分だけを残す ---
    df = dedup_latest(df, business_key)

    # --- 3b. ソースをまたぐ結合（設定がある場合だけ） ---
    # ブロンズ層に分かれて届いた同じ実体を、業務の単位に組み立て直す。
    # 結合相手も同じ規則で最新化してから左結合する（左結合なので、
    # 人事側に行がない社員も落ちない）。
    for join_bronze, join_key in ENRICHMENTS.get(silver, []):
        jdf = spark.table(bronze_ref(join_bronze))
        jlatest = jdf.agg(F.max("ingest_date").alias("d")).collect()[0]["d"]
        jdf = dedup_latest(jdf.filter(F.col("ingest_date") == F.lit(jlatest)), join_key)
        jdf = jdf.drop(*[c for c in AUDIT_COLS if c in jdf.columns])
        before = df.count()
        df = df.join(jdf, on=join_key, how="left")
        unmatched = df.filter(F.col(jdf.columns[-1]).isNull()).count() if before else 0
        if unmatched:
            print(f"[WARN] {silver}: {join_bronze} と結合できない行が {unmatched} 件あります"
                  f"（結合キー: {'+'.join(join_key)}）。人事側の未登録者が含まれていないか確認してください。")
        scope += f" + {join_bronze}"

    # --- 4. エンティティ型キーを業務キーから生成する ---
    # 業務キーで重複排除済みなので、生成したキーは構造的に一意になる。
    # ソースが ID を振り直しても、同じ実体は同じキーを持ち続ける。
    if generate_key:
        df = df.withColumn(
            entity_key,
            F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in business_key]), 256),
        )

    # --- 5. 「いつ時点のデータか」を残す ---
    # _valid_as_of は技術的なメタデータ。全テーブルに付けるが、
    # オントロジーには原則バインドしない（アンダースコア始まりの名前も業務語彙に不適切）。
    df = df.withColumn("_valid_as_of", F.col("ingest_date").cast("string"))

    # 稼働情報だけは鮮度が判断に直結するため、業務語彙として名前を付けた列を別途持たせる。
    # 「その空き工数はいつ時点の情報か」をエージェントに聞けるようにするため。
    if silver in EXPOSE_AS_OF_DATE:
        df = df.withColumn("DataAsOfDate", F.col("ingest_date").cast("string"))

    df = df.drop(*[c for c in AUDIT_COLS if c in df.columns])

    # --- 5b. 列順を固定する（結合すると列順がソース依存になるため） ---
    if silver in COLUMN_ORDER:
        ordered = COLUMN_ORDER[silver]
        df = df.select(*ordered, *[c for c in df.columns if c not in ordered])

    df = df.cache()
    silver_rows = df.count()
    valid_as_of = df.agg(F.max("_valid_as_of").alias("d")).collect()[0]["d"]

    # --- 6. 手順2の検証結果で、公開するかどうかを決める ---
    if n_dup == 0:
        df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(silver)
        report.append((silver, mode, scope, bronze_rows, silver_rows, valid_as_of, "OK", ""))
        key_note = "生成キー" if generate_key else "業務キー"
        print(f"[OK]   {silver:28s} {mode:11s} {bronze_rows:6d} -> {silver_rows:6d} rows  ({key_note}: {entity_key})")
    else:
        # 公開しない。前回の正常なシルバー層をそのまま残す。
        quarantine = f"dq_{silver[3:]}_duplicates"
        (
            quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
            .format("delta").saveAsTable(quarantine)
        )
        existed = spark.catalog.tableExists(silver)
        note = (
            f"同一取り込み日の中で業務キー {'+'.join(business_key)} が {n_dup} 種類で重複"
            f"（ソース側の実データ異常）。"
            + ("前回のシルバー層を維持。" if existed else "シルバー層は未作成。")
            + f" 重複行を {quarantine} に隔離。"
        )
        report.append((silver, mode, scope, bronze_rows, silver_rows, valid_as_of, "NG", note))
        print(f"[NG]   {silver:28s} 公開を見送りました — {note}")

    df.unpersist()

#### 状態テーブルを出力する

公開を見送ったテーブルは古いデータのまま残ります。それに気づけるよう、各テーブルの
鮮度（`_valid_as_of`）と状態を `dq_silver_status` に常に可視化しておきます。

In [ ]:
# ============================================================
# 状態テーブルを毎回出力する（dq_silver_status）
# ============================================================
# 公開を見送ったテーブルは古いデータのまま残ります。それに気づけるよう、
# 各テーブルの鮮度（_valid_as_of）と状態を常に可視化しておきます。
schema = ["silver_table", "mode", "scope", "bronze_rows", "silver_rows", "valid_as_of", "status", "note"]
status_df = (
    spark.createDataFrame(report, schema)
    .withColumn("checked_at", F.current_timestamp())
    .withColumn("age_days", F.datediff(F.current_date(), F.to_date("valid_as_of")))
    .withColumn(
        "freshness",
        F.when(F.col("age_days") <= STALE_AFTER_DAYS, F.lit("FRESH")).otherwise(F.lit("STALE")),
    )
)
status_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dq_silver_status")

ng = [r for r in report if r[6] == "NG"]
stale = [r for r in status_df.filter(F.col("freshness") == "STALE").collect()]
SILVER_BUILD_STATUS = "WARN" if ng else "OK"

print()
print("=" * 78)
if ng:
    print(f"SILVER_BUILD_STATUS = WARN  —  {len(ng)} / {len(ENTITIES)} テーブルの公開を見送りました")
    print()
    print("【WARN が出たときのアクション】")
    print("  エンティティ型キーは業務キーから生成しているため、通常の取り込みでは")
    print("  この警告は出ません。出たということは、ソース側で業務キーが重複しています。")
    print()
    print("  1. dq_<entity>_duplicates を開き、重複している行を確認する")
    print("  2. 実データの異常か（同じ人が二重登録されている等）を判断する")
    print("     → 異常ならソースシステムの管理者に連携し、修正後に再実行")
    print("  3. 異常でなければ、業務キーの定義が実態と合っていない")
    print("     → ENTITIES の業務キーに列を足して再実行")
    print("       （例: 同じ人が複数所属を持つなら PersonId + OrganizationId）")
    print()
    print("  公開を見送ったテーブルはオントロジーから見て前回の状態のままです。")
    print("  dq_silver_status の freshness 列で、どれだけ古いかを確認できます。")
else:
    print(f"SILVER_BUILD_STATUS = OK  —  全 {len(ENTITIES)} テーブルを公開しました")

if stale:
    print()
    print(f"[鮮度] {len(stale)} テーブルが {STALE_AFTER_DAYS} 日以上更新されていません:")
    for r in stale:
        print(f"  - {r['silver_table']}: {r['valid_as_of']} 時点（{r['age_days']} 日前）")
print("=" * 78)
print()
display(status_df.select("silver_table", "mode", "silver_rows", "valid_as_of", "age_days", "freshness", "status", "note"))


# ============================================================
# 定義とテーブル一覧を一致させる
# ============================================================
# シルバー層の sv_* は、オントロジーのエンティティ型と1対1で対応します。
# ENTITIES から外したエンティティのテーブルは上書きされないため、放っておくと
# 残り続け、「オントロジーがバインドしていない sv_*」がシルバー層に並びます。
# dq_<entity>_duplicates も、重複が解消された後は不要になります。
def silver_table_names():
    try:
        return [t.name for t in spark.catalog.listTables()]
    except Exception:
        return [r["tableName"] for r in spark.sql("SHOW TABLES").collect()]


expected = set(ENTITIES) | {"dq_silver_status"}
# 今回の実行で隔離テーブルを出したものは残す（中身の確認が必要なため）
expected |= {f"dq_{s[3:]}_duplicates" for s in ENTITIES if s in
             {r[0] for r in report if r[6] == "NG"}}

orphans = sorted(
    t for t in silver_table_names()
    if (t.startswith("sv_") or t.startswith("dq_")) and t not in expected
)

print()
if not orphans:
    print(f"テーブル一覧は定義と一致しています"
          f"（sv_* {len(ENTITIES)} 本 + dq_silver_status）。")
else:
    print(f"[整理] 定義に無いテーブルが {len(orphans)} 本あります:")
    for t in orphans:
        kind = "オントロジーがバインドしていない sv_*" if t.startswith("sv_") \
            else "解消済みの隔離テーブル"
        print(f"  - {t}（{kind}）")

    if DROP_ORPHAN_TABLES:
        for t in orphans:
            spark.sql(f"DROP TABLE IF EXISTS {t}")
        print(f"\n{len(orphans)} 本を削除しました。")
    else:
        print("\nDROP_ORPHAN_TABLES = False のため削除していません。")
        print("削除する場合は True にして再実行してください。")

n_sv = len([t for t in silver_table_names() if t.startswith("sv_")])
print(f"\nシルバー層の sv_* テーブル: {n_sv} 本"
      + ("  ← オントロジーのエンティティ型17個と一致" if n_sv == len(ENTITIES)
         else f"  ← 想定は {len(ENTITIES)} 本"))

# パイプラインから呼ぶ場合は、この値で後続処理を分岐させる
# mssparkutils.notebook.exit(SILVER_BUILD_STATUS)

#### 動作確認

Step 6 のシナリオ1「製造業向けの生成AI案件を経験し、Fabric または Databricks の
スキルを持ち、2026年9月に30%以上の空き工数があるメンバー」を SQL で再現します。
**3名がヒットすれば正常**です。

In [ ]:
# ============================================================
# 動作確認: 要員探索シナリオ
# ============================================================
# 「製造業向けの生成AI案件を経験し、Fabric または Databricks のスキルを持ち、
#   2026年9月に30%以上の空き工数があるメンバー」
check = spark.sql("""
SELECT p.PersonId, p.FullName, p.JobTitle, p.OrganizationName, av.AvailablePercent
FROM sv_person p
JOIN sv_assignment a    ON a.PersonId  = p.PersonId
JOIN sv_project pj      ON pj.ProjectId = a.ProjectId
JOIN sv_customer c      ON c.CustomerId = pj.CustomerId
JOIN sv_person_skill ps ON ps.PersonId = p.PersonId
JOIN sv_availability av ON av.PersonId = p.PersonId
WHERE c.Industry = '製造'
  AND pj.SolutionDomain = 'AI'
  AND ps.SkillId IN ('SK001', 'SK002')
  AND av.YearMonth = '2026-09'
  AND av.AvailablePercent >= 30
GROUP BY p.PersonId, p.FullName, p.JobTitle, p.OrganizationName, av.AvailablePercent
ORDER BY av.AvailablePercent DESC
""")
print(f"\n要員探索シナリオのヒット件数: {check.count()} 件（3名が想定）")
display(check)